# KDIC 검색 방식 5단계 비교·평가 노트북 — Dual Hybrid 정밀 가중치 실험판

`KDIC_output`의 동일한 `chunks.jsonl`과 `Evaluation_DataSet_v3*.xlsx`의
`평가데이터셋 v3` 시트를 사용하고, 검색 방식만 단계적으로 비교합니다.

| 단계 | 고정 조건 | 비교 대상 |
|---|---|---|
| 1차 | BM25 검색 공식 | Standard, Nori-none, Nori-discard, Nori-mixed |
| 2차 | 1차 Best Analyzer | BM25, BM25F, BGE-M3 Sparse |
| 3차 | BGE-M3 모델 | Dense-content, Dense-structured |
| 4-1 | RRF_K=60, depth=30 | Dense 0.60~0.95 / Sparse 0.40~0.05, 0.05 간격 |
| 4-1A | Dense-structured 고정 | BGE-M3 Sparse와 결합한 8개 가중치 |
| 4-1B | Dense-structured 고정 | BM25 Nori-none/discard 중 우수 방식과 결합한 8개 가중치 |
| 4-2 | 각 Hybrid 계열의 Best 가중치 | depth 20, 30, 50 비교 |
| 5차 | 단일 검색기 + 계열별 Best Hybrid | Dense, BGE Sparse, BM25, Hybrid-BGE, Hybrid-BM25 |

모든 검색 방식이 RAG에 최종 반환하는 청크는 **Top 5**로 고정합니다.


## 실험 규칙

1. 최종 `retrieved_chunk_ids`와 컨텍스트에는 모든 방식에서 5개 청크만 사용합니다.
2. `MRR@10`과 `MAP@10` 계산을 위해 평가용 순위는 내부적으로 10위까지 보존합니다.
3. Hybrid의 `depth=20/30/50`은 RRF 결합 전 각 검색기의 후보 수이며, 결합 후 결과는 항상 Top 5입니다.
4. 선정 기준은 `nDCG@5` → `Recall@5` → `MRR@10` → 지연시간 순입니다.
5. `split=validation/test`가 없으면 평가 포함 문항 전체로 승자를 선정하며 예비 실험으로 해석합니다.
6. 입력은 `Evaluation_DataSet_v3*.xlsx`의 `평가데이터셋 v3` 시트를 사용합니다.
7. `evaluation_id`만 문항 고유키로 사용합니다. 도메인별로 재사용되는 `질문ID`는 중복을 허용합니다.
8. 현재 검수 엑셀 전체를 평가 모집단으로 간주하므로 `gold_review_status`는 기록만 보존하고 필터에는 사용하지 않습니다.
9. 유효한 `gold_chunk_ids`는 이진 관련성 평가의 권위 필드로 유지합니다.
10. 손상된 `gold_chunk_ids`만 Primary와 Supporting의 합집합으로 복구합니다.
11. 중첩 JSON 배열은 1차원 배열로 평탄화하고, 모든 복구·불일치를 검증 CSV에 기록합니다.
12. `gold_evidence_requirement` 누락은 가능한 경우 `PRIMARY_ONLY`로 추론합니다.
13. 유효한 Gold가 없거나 실제 청크에 없는 ID를 참조하는 행만 자동 제외합니다.
14. `nDCG@5`는 Primary=2, Supporting 및 미등급 Gold=1로 계산합니다.
15. `Complete@5`는 `gold_evidence_requirement`에 따라 필요한 근거 집합 전체가 Top 5에 들어왔는지 평가합니다.
16. BGE-M3 Sparse는 `structured` 입력, Dense-content는 본문만, Dense-structured는 제목+소제목+본문을 사용합니다.
17. Hybrid의 Dense 축은 **BGE-M3 Dense-structured로 고정**합니다.
18. Hybrid의 Sparse 축은 **BGE-M3 Sparse-structured**와 **BM25 Nori-none/discard 중 우수 방식**을 별도 계열로 비교합니다.
19. Hybrid 가중치는 Dense 0.60~0.95, Sparse 0.40~0.05를 0.05 간격으로 비교합니다.
20. 각 Sparse 계열이 반드시 최종 비교까지 남도록 4-1차에서 계열별 Best 가중치를 각각 선정합니다.
21. Hybrid는 가중 RRF를 사용하며 `RRF_K=60`으로 고정합니다.
22. 인덱스 생성과 문서 임베딩 시간은 질의 지연시간에서 제외합니다.

평가 지표: `Hit@3`, `Recall@5`, `Primary Recall@5`, `MRR@10`, `MAP@10`,
`Complete@5`, `nDCG@5`, `Precision@5`, `Useful Precision@5`, `F1@5`,
검색 지연시간, 컨텍스트 한도 초과율.

답변 생성 LLM, 질의 재작성, 재랭커, 의도 가중치, 청크 타입 가중치는 사용하지 않습니다.


## 0. 패키지 설치


In [ ]:

%pip install -q \
  "elasticsearch==8.17.2" \
  "FlagEmbedding==1.4.0" \
  "transformers==4.57.6" \
  "tiktoken==0.8.0" \
  "pandas>=2.2,<3" \
  "openpyxl==3.1.5" \
  "numpy>=1.26,<3" \
  "scipy>=1.12,<2" \
  "matplotlib>=3.8,<4" \
  "seaborn>=0.13,<1"

import importlib.metadata as _package_metadata

for _name, _expected in {
    "FlagEmbedding": "1.4.0",
    "transformers": "4.57.6",
    "openpyxl": "3.1.5",
}.items():
    _installed = _package_metadata.version(_name)
    if _installed != _expected:
        raise RuntimeError(
            f"{_name} 버전 불일치: expected={_expected}, installed={_installed}"
        )
    print(f"{_name}: {_installed}")


## 1. Elasticsearch와 Nori 준비

Colab에서는 Elasticsearch 8.17.2를 내려받고 동일 버전의 `analysis-nori`를 설치합니다.
Colab cgroup 문제를 피하는 JVM 옵션과 실제 Nori 토큰화 사전검사를 포함합니다.


In [ ]:

from __future__ import annotations

import os
import pwd
import shutil
import subprocess
import tarfile
import time
import urllib.request
from pathlib import Path

import requests

IS_COLAB = "google.colab" in __import__("sys").modules
START_LOCAL_ELASTICSEARCH = IS_COLAB
ES_VERSION = "8.17.2"
ES_URL = "http://127.0.0.1:9200"
ES_USERNAME = None
ES_PASSWORD = None

def start_colab_elasticsearch(version: str) -> None:
    install_root = Path("/content") if Path("/content").exists() else Path.cwd()

    archive_path = (
        install_root
        / f"elasticsearch-{version}-linux-x86_64.tar.gz"
    )
    es_home = install_root / f"elasticsearch-{version}"
    es_user = "kdic_es"

    # 이미 Elasticsearch가 실행 중인지 확인
    try:
        response = requests.get(ES_URL, timeout=2)
        response.raise_for_status()

        print(
            "Elasticsearch가 이미 실행 중입니다:",
            response.json()["version"]["number"],
        )
        return

    except Exception:
        pass

    # Elasticsearch 다운로드 및 압축 해제
    if not es_home.exists():
        url = (
            "https://artifacts.elastic.co/downloads/elasticsearch/"
            f"{archive_path.name}"
        )

        if not archive_path.exists():
            print("Elasticsearch 다운로드:", url)
            urllib.request.urlretrieve(url, archive_path)

        print("Elasticsearch 압축 해제 중...")

        with tarfile.open(archive_path, "r:gz") as archive:
            archive.extractall(install_root)

    # 실행용 사용자 생성
    try:
        pwd.getpwnam(es_user)
        print("기존 Elasticsearch 사용자 사용:", es_user)

    except KeyError:
        print("Elasticsearch 사용자 생성:", es_user)

        subprocess.run(
            ["useradd", "-m", es_user],
            check=True,
        )

    # Elasticsearch 설치 디렉터리 소유권 변경
    subprocess.run(
        [
            "chown",
            "-R",
            f"{es_user}:{es_user}",
            str(es_home),
        ],
        check=True,
    )

    # Nori 플러그인 설치 여부 확인
    plugin_bin = es_home / "bin" / "elasticsearch-plugin"

    plugin_list_result = subprocess.run(
        [
            "runuser",
            "-u",
            es_user,
            "--",
            str(plugin_bin),
            "list",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    installed_plugins = plugin_list_result.stdout

    if "analysis-nori" not in installed_plugins:
        print("analysis-nori 플러그인 설치 중...")

        subprocess.run(
            [
                "runuser",
                "-u",
                es_user,
                "--",
                str(plugin_bin),
                "install",
                "--batch",
                "analysis-nori",
            ],
            check=True,
        )

        print("analysis-nori 플러그인 설치 완료")

    else:
        print("analysis-nori 플러그인이 이미 설치되어 있습니다.")

    # 데이터 및 로그 경로 생성
    data_dir = install_root / "kdic-es-data"
    logs_dir = install_root / "kdic-es-logs"

    data_dir.mkdir(exist_ok=True)
    logs_dir.mkdir(exist_ok=True)

    # kdic_es 사용자가 쓸 수 있도록 소유권 변경
    subprocess.run(
        [
            "chown",
            "-R",
            f"{es_user}:{es_user}",
            str(data_dir),
            str(logs_dir),
        ],
        check=True,
    )

    # Elasticsearch 설정 작성
    config_path = es_home / "config" / "elasticsearch.yml"

    config_path.write_text(
        "\n".join(
            [
                "cluster.name: kdic-retrieval-eval",
                "node.name: kdic-node-1",
                "discovery.type: single-node",
                "xpack.security.enabled: false",
                "xpack.security.autoconfiguration.enabled: false",
                "xpack.ml.enabled: false",
                "node.roles: [master, data, ingest]",
                "network.host: 127.0.0.1",
                "http.port: 9200",
                f"path.data: {data_dir}",
                f"path.logs: {logs_dir}",
                "",
            ]
        ),
        encoding="utf-8",
    )

    subprocess.run(
        [
            "chown",
            f"{es_user}:{es_user}",
            str(config_path),
        ],
        check=True,
    )

    # 중요: /content 바로 아래가 아니라 쓰기 가능한 logs_dir 내부에 생성
    pid_path = logs_dir / "elasticsearch.pid"

    if pid_path.exists():
        pid_path.unlink()

    command = [
        "runuser",
        "-u",
        es_user,
        "--",
        "env",
        # Colab의 /proc/self/cgroup 경로가 실제 마운트 범위를 벗어나는
        # 문제를 피하기 위해 cgroup v2 루트로 강제합니다.
        "ES_JAVA_OPTS=-Xms1g -Xmx1g -Des.cgroups.hierarchy.override=/",
        str(es_home / "bin" / "elasticsearch"),
        "-d",
        "-p",
        str(pid_path),
    ]

    cgroup_cpu_stat = Path("/sys/fs/cgroup/cpu.stat")
    if cgroup_cpu_stat.exists():
        try:
            cgroup_cpu_stat.read_text(encoding="utf-8")
            print("Colab cgroup 루트 확인: 읽기 가능")
        except Exception as error:
            print("Colab cgroup 루트 확인 경고:", error)
    else:
        print("Colab cgroup 루트 확인 경고: cpu.stat 없음")

    print("Elasticsearch 실행 중...")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print("\nElasticsearch 실행 실패")
        print("종료 코드:", result.returncode)

        if result.stdout.strip():
            print("\n[stdout]")
            print(result.stdout)

        if result.stderr.strip():
            print("\n[stderr]")
            print(result.stderr)

        print("\n로그 디렉터리:", logs_dir)

        log_files = sorted(logs_dir.glob("*.log"))

        if log_files:
            print("생성된 로그 파일:")
            for log_file in log_files:
                print("-", log_file)

        raise RuntimeError(
            f"Elasticsearch 프로세스 실행 실패: "
            f"종료 코드 {result.returncode}"
        )

    # Elasticsearch HTTP 서버 준비 대기
    for attempt in range(90):
        try:
            response = requests.get(
                ES_URL,
                timeout=2,
            )

            if response.ok:
                version_number = response.json()["version"]["number"]

                print(
                    "Elasticsearch 시작 완료:",
                    version_number,
                )
                print("PID 파일:", pid_path)
                return

        except Exception:
            pass

        if attempt % 10 == 0:
            print(
                f"Elasticsearch 시작 대기 중... "
                f"{attempt * 2}초"
            )

        time.sleep(2)

    # 시작되지 않았다면 로그 확인
    print("Elasticsearch 로그 확인:")

    for log_file in sorted(logs_dir.glob("*.log")):
        print("\n로그 파일:", log_file)

        try:
            log_lines = log_file.read_text(
                encoding="utf-8",
                errors="replace",
            ).splitlines()

            print("\n".join(log_lines[-100:]))

        except Exception as error:
            print("로그 읽기 실패:", error)

    raise RuntimeError(
        f"Elasticsearch 시작 실패. "
        f"로그를 확인하세요: {logs_dir}"
    )


# 함수 정의만 하고 넘어가지 않도록 이 셀에서 실제로 실행합니다.
if START_LOCAL_ELASTICSEARCH:
    start_colab_elasticsearch(ES_VERSION)
else:
    print("외부 Elasticsearch 사용:", ES_URL)


## 2. 공통 평가 모듈 생성


In [ ]:
%%writefile kdic_retrieval_eval.py
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import statistics
import time
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy import sparse


REQUIRED_EVAL_COLUMNS = {
    "evaluation_id",
    "question",
    "gold_source_urls",
    "gold_primary_chunk_ids",
    "gold_supporting_chunk_ids",
    "gold_chunk_ids",
    "gold_evidence_requirement",
    "multi_chunk_required",
}

METRIC_COLUMNS = [
    "hit_at_3",
    "recall_at_5",
    "primary_recall_at_5",
    "mrr_at_10",
    "ap_at_10",
    "complete_at_5",
    "ndcg_at_5",
    "precision_at_5",
    "useful_precision_at_5",
    "f1_at_5",
]

BUSINESS_CODE_TO_LABEL = {
    "deposit_protection": "예금자보호제도",
    "deposit_insurance_payout": "예금보험금",
    "unclaimed_funds": "고객 미수령금",
    "mistaken_transfer": "착오송금 반환지원",
    "debt_adjustment": "채무조정",
    "hidden_assets_report": "은닉재산 신고",
}
BUSINESS_LABEL_TO_CODE = {
    label: code for code, label in BUSINESS_CODE_TO_LABEL.items()
}


@dataclass(frozen=True)
class ExperimentConfig:
    final_top_k: int = 5
    evaluation_depth: int = 10
    complete_max_required: int = 5
    context_budget_tokens: int = 8_000
    context_top_k: int = 5
    selection_metric: str = "ndcg_at_5"
    selection_split: str = "validation"
    split_column: str = "split"
    business_filter_mode: str = "none"
    business_filter_column: str = "predicted_business_function"
    latency_repeats: int = 1
    rrf_k: int = 60
    allowed_review_statuses: tuple[str, ...] | None = None
    evaluation_sheet_name: str = "평가데이터셋 v3"
    gold_normalization_policy: str = "gold_chunk_ids_authoritative"


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL 파싱 실패: {path}, line={line_number}") from exc
    return rows


def safe_extract_zip(zip_path: Path, destination: Path) -> Path:
    destination = destination.resolve()
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            resolved = (destination / member.filename).resolve()
            if destination not in resolved.parents and resolved != destination:
                raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
        archive.extractall(destination)
    return destination


def find_unique(root: Path, filename: str, *, required: bool = True) -> Path | None:
    matches = sorted(root.rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(f"{root} 아래에서 {filename}을 찾지 못했습니다.")
        return None
    if len(matches) > 1:
        raise RuntimeError(f"{filename}이 여러 개입니다: {matches}")
    return matches[0]


def _is_missing(value: Any) -> bool:
    return value is None or (isinstance(value, float) and math.isnan(value)) or not str(value).strip()


def _ordered_unique(values: Sequence[str]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for value in values:
        normalized = str(value).strip()
        if normalized and normalized not in seen:
            seen.add(normalized)
            output.append(normalized)
    return output


def parse_json_array_strict(value: Any) -> tuple[list[str], str | None]:
    """JSON 배열을 읽고 실수로 생긴 중첩 배열은 1차원으로 복구한다."""
    if _is_missing(value):
        return [], None
    if isinstance(value, list):
        parsed = value
    else:
        text = str(value).strip()
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:
            return [], f"JSON_PARSE_ERROR: {exc.msg}"
    if not isinstance(parsed, list):
        return [], f"NOT_JSON_ARRAY: {type(parsed).__name__}"

    flattened: list[str] = []
    stack = list(parsed)
    while stack:
        item = stack.pop(0)
        if isinstance(item, list):
            stack = list(item) + stack
            continue
        if isinstance(item, (dict, tuple, set)):
            return [], f"UNSUPPORTED_ARRAY_ITEM: {type(item).__name__}"
        flattened.append(str(item))
    return _ordered_unique(flattened), None


def load_chunks(data_zip: Path, work_dir: Path) -> tuple[list[dict[str, Any]], Path]:
    extracted = safe_extract_zip(data_zip, work_dir / "extracted_kdic_output")
    chunks_path = find_unique(extracted, "chunks.jsonl")
    chunks = read_jsonl(chunks_path)
    if not chunks:
        raise ValueError("chunks.jsonl이 비어 있습니다.")
    chunk_ids = [str(row.get("chunk_id", "")).strip() for row in chunks]
    if any(not chunk_id for chunk_id in chunk_ids):
        raise ValueError("빈 chunk_id가 있습니다.")
    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError("중복 chunk_id가 있습니다.")
    return chunks, chunks_path


EVAL_COLUMN_ALIASES = {
    "예상질문": "question",
    "질문ID": "question_id",
    "도메인": "domain",
    "질문 복잡도": "question_complexity",
    "중요도": "priority",
}


def load_evaluation_xlsx(path: Path, *, sheet_name: str) -> pd.DataFrame:
    available_sheets = pd.ExcelFile(path, engine="openpyxl").sheet_names
    if sheet_name not in available_sheets:
        raise ValueError(
            f"평가 시트 '{sheet_name}'가 없습니다. 사용 가능한 시트={available_sheets}"
        )
    frame = pd.read_excel(
        path,
        sheet_name=sheet_name,
        dtype=object,
        engine="openpyxl",
    )
    frame.columns = [str(column).strip() for column in frame.columns]
    rename_map = {
        source: target
        for source, target in EVAL_COLUMN_ALIASES.items()
        if source in frame.columns and target not in frame.columns
    }
    frame = frame.rename(columns=rename_map)
    missing = REQUIRED_EVAL_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"평가 Excel 필수 열이 없습니다: {sorted(missing)}")

    frame = frame.dropna(how="all").reset_index(drop=True)
    frame.insert(0, "_source_excel_row", np.arange(2, len(frame) + 2))
    frame["evaluation_id"] = frame["evaluation_id"].fillna("").astype(str).str.strip()
    if frame["evaluation_id"].eq("").any():
        raise ValueError("빈 evaluation_id가 있습니다.")
    if frame["evaluation_id"].duplicated().any():
        duplicated = frame.loc[
            frame["evaluation_id"].duplicated(False), "evaluation_id"
        ].tolist()
        raise ValueError(f"evaluation_id가 중복되었습니다: {duplicated[:10]}")

    frame["question"] = frame["question"].fillna("").astype(str).str.strip()
    if frame["question"].eq("").any():
        ids = frame.loc[frame["question"].eq(""), "evaluation_id"].tolist()
        raise ValueError(f"빈 question이 있습니다: {ids[:10]}")
    if "question_id" in frame.columns:
        frame["question_id"] = (
            frame["question_id"].fillna("").astype(str).str.strip()
        )
    return frame


def prepare_evaluation_rows(
    eval_df: pd.DataFrame,
    *,
    chunks: Sequence[Mapping[str, Any]],
    config: ExperimentConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if config.gold_normalization_policy != "gold_chunk_ids_authoritative":
        raise ValueError(
            "gold_normalization_policy는 gold_chunk_ids_authoritative만 지원합니다."
        )

    chunks_by_id = {
        str(chunk["chunk_id"]).strip(): chunk
        for chunk in chunks
    }
    corpus_ids = set(chunks_by_id)
    allowed = (
        {str(value).strip().lower() for value in config.allowed_review_statuses}
        if config.allowed_review_statuses
        else None
    )
    included_records: list[dict[str, Any]] = []
    excluded_records: list[dict[str, Any]] = []
    report_records: list[dict[str, Any]] = []

    for record in eval_df.to_dict(orient="records"):
        evaluation_id = str(record["evaluation_id"]).strip()
        primary, primary_error = parse_json_array_strict(
            record.get("gold_primary_chunk_ids")
        )
        supporting, supporting_error = parse_json_array_strict(
            record.get("gold_supporting_chunk_ids")
        )
        supplied_gold, gold_error = parse_json_array_strict(
            record.get("gold_chunk_ids")
        )
        supplied_urls, url_error = parse_json_array_strict(
            record.get("gold_source_urls")
        )
        notes: list[str] = []
        warnings_: list[str] = []
        exclusion_reasons: list[str] = []

        for field_name in (
            "gold_primary_chunk_ids",
            "gold_supporting_chunk_ids",
            "gold_chunk_ids",
        ):
            raw_value = record.get(field_name)
            if isinstance(raw_value, str) and raw_value.strip().startswith("[["):
                notes.append(f"{field_name}_nested_array_flattened")

        if primary_error or supporting_error:
            if supplied_gold and not gold_error:
                primary = list(supplied_gold)
                supporting = []
                notes.append("graded_parse_error_fallback_to_gold")
            else:
                exclusion_reasons.append("graded_gold_parse_error")

        if supplied_gold and not gold_error:
            canonical_gold = list(supplied_gold)
        elif primary or supporting:
            canonical_gold = _ordered_unique([*primary, *supporting])
            notes.append("gold_rebuilt_from_primary_supporting")
        else:
            canonical_gold = []
            exclusion_reasons.append("no_valid_gold")

        graded_union = _ordered_unique([*primary, *supporting])
        graded_not_in_gold = [
            chunk_id for chunk_id in graded_union
            if chunk_id not in canonical_gold
        ]
        gold_without_grade = [
            chunk_id for chunk_id in canonical_gold
            if chunk_id not in graded_union
        ]
        if graded_not_in_gold:
            warnings_.append("graded_not_in_gold")
        if gold_without_grade:
            warnings_.append("gold_without_grade")

        unknown_ids = sorted(
            set([*primary, *supporting, *canonical_gold]) - corpus_ids
        )
        if unknown_ids:
            exclusion_reasons.append("unknown_chunk_ids")

        raw_requirement = record.get("gold_evidence_requirement")
        requirement = (
            ""
            if _is_missing(raw_requirement)
            else str(raw_requirement).strip().upper()
        )
        if requirement not in {"PRIMARY_ONLY", "PRIMARY_PLUS_SUPPORT", "ALL"}:
            if primary and not supporting and set(canonical_gold) == set(primary):
                requirement = "PRIMARY_ONLY"
                notes.append("evidence_requirement_inferred_primary_only")
            else:
                exclusion_reasons.append("invalid_evidence_requirement")

        if requirement == "PRIMARY_ONLY":
            evidence_ids = list(primary)
        elif requirement == "PRIMARY_PLUS_SUPPORT":
            evidence_ids = _ordered_unique([*primary, *supporting])
        elif requirement == "ALL":
            evidence_ids = list(canonical_gold)
        else:
            evidence_ids = []

        expected_urls: list[str] = []
        for chunk_id in evidence_ids:
            source_url = str(
                chunks_by_id.get(chunk_id, {}).get("source_url", "")
            ).strip()
            if source_url and source_url not in expected_urls:
                expected_urls.append(source_url)
        if url_error:
            warnings_.append("gold_source_urls_parse_error")
        elif supplied_urls != expected_urls:
            warnings_.append("gold_source_urls_mismatch")

        raw_business_function = record.get("gold_business_function")
        business_function = (
            ""
            if _is_missing(raw_business_function)
            else str(raw_business_function).strip()
        )
        if not business_function:
            raw_domain = record.get("domain")
            domain = (
                ""
                if _is_missing(raw_domain)
                else str(raw_domain).strip()
            )
            inferred = BUSINESS_LABEL_TO_CODE.get(domain, "")
            if inferred:
                business_function = inferred
                notes.append("business_function_inferred_from_domain")
            else:
                exclusion_reasons.append("missing_business_function")

        raw_review_status = record.get("gold_review_status")
        review_status = (
            ""
            if _is_missing(raw_review_status)
            else str(raw_review_status).strip().lower()
        )
        if allowed is not None and review_status not in allowed:
            exclusion_reasons.append("review_status_not_allowed")

        raw_multi_value = record.get("multi_chunk_required")
        multi_value = (
            ""
            if _is_missing(raw_multi_value)
            else str(raw_multi_value).strip().upper()
        )
        if multi_value not in {"Y", "N"}:
            warnings_.append("invalid_multi_chunk_required")
        elif (multi_value == "Y") != (len(evidence_ids) > 1):
            warnings_.append("multi_chunk_flag_mismatch")

        normalized = dict(record)
        normalized["gold_business_function"] = business_function
        normalized["gold_evidence_requirement"] = requirement
        normalized["gold_primary_chunk_ids"] = primary
        normalized["gold_supporting_chunk_ids"] = supporting
        normalized["gold_chunk_ids"] = canonical_gold
        normalized["gold_source_urls"] = supplied_urls
        normalized["gold_normalization_status"] = (
            "normalized" if notes else "unchanged"
        )
        normalized["gold_normalization_notes"] = "|".join(notes)
        normalized["gold_validation_warnings"] = "|".join(warnings_)

        included = not exclusion_reasons
        report_records.append(
            {
                "_source_excel_row": record.get("_source_excel_row"),
                "evaluation_id": evaluation_id,
                "question_id": record.get("question_id", ""),
                "question": record.get("question", ""),
                "gold_review_status": record.get("gold_review_status", ""),
                "included": included,
                "exclusion_reason": "|".join(exclusion_reasons),
                "normalization_status": normalized[
                    "gold_normalization_status"
                ],
                "normalization_notes": normalized[
                    "gold_normalization_notes"
                ],
                "validation_warnings": normalized[
                    "gold_validation_warnings"
                ],
                "primary_parse_error": primary_error or "",
                "supporting_parse_error": supporting_error or "",
                "gold_parse_error": gold_error or "",
                "source_urls_parse_error": url_error or "",
                "unknown_chunk_ids": json.dumps(
                    unknown_ids, ensure_ascii=False
                ),
                "graded_not_in_gold": json.dumps(
                    graded_not_in_gold, ensure_ascii=False
                ),
                "gold_without_grade": json.dumps(
                    gold_without_grade, ensure_ascii=False
                ),
                "expected_gold_source_urls": json.dumps(
                    expected_urls, ensure_ascii=False
                ),
                "supplied_gold_source_urls": json.dumps(
                    supplied_urls, ensure_ascii=False
                ),
                "normalized_primary_chunk_ids": json.dumps(
                    primary, ensure_ascii=False
                ),
                "normalized_supporting_chunk_ids": json.dumps(
                    supporting, ensure_ascii=False
                ),
                "normalized_gold_chunk_ids": json.dumps(
                    canonical_gold, ensure_ascii=False
                ),
            }
        )
        if included:
            included_records.append(normalized)
        else:
            excluded = dict(normalized)
            excluded["exclusion_reason"] = "|".join(exclusion_reasons)
            excluded_records.append(excluded)

    included_df = pd.DataFrame(included_records)
    excluded_df = pd.DataFrame(excluded_records)
    report_df = pd.DataFrame(report_records)
    if included_df.empty:
        raise RuntimeError("Gold 검증을 통과한 평가 문항이 없습니다.")
    return (
        included_df.reset_index(drop=True),
        excluded_df.reset_index(drop=True),
        report_df,
    )


def validate_gold_chunk_ids(eval_df: pd.DataFrame, chunks: Sequence[Mapping[str, Any]]) -> None:
    corpus_ids = {str(chunk["chunk_id"]) for chunk in chunks}
    unknown: dict[str, list[str]] = {}
    for row in eval_df.itertuples(index=False):
        gold = set(row.gold_chunk_ids) | set(row.gold_primary_chunk_ids) | set(row.gold_supporting_chunk_ids)
        missing = sorted(gold - corpus_ids)
        if missing:
            unknown[str(row.evaluation_id)] = missing
    if unknown:
        preview = dict(list(unknown.items())[:10])
        raise ValueError(f"Gold 청크가 corpus에 없습니다. 예시={preview}, 전체 문항 수={len(unknown)}")


def build_content_text(chunk: Mapping[str, Any]) -> str:
    return str(chunk.get("content") or "").strip()


def build_structured_text(chunk: Mapping[str, Any]) -> str:
    parts: list[str] = []
    for field, label in (
        ("title", "제목"),
        ("section_title", "소제목"),
        ("content", "본문"),
    ):
        value = str(chunk.get(field) or "").strip()
        if value:
            parts.append(f"{label}: {value}")
    return "\n".join(parts)


def build_search_text(chunk: Mapping[str, Any]) -> str:
    """BM25/BM25F 및 기존 호출부용 구조화 텍스트."""
    return build_structured_text(chunk)


TEXT_BUILDERS: dict[str, Callable[[Mapping[str, Any]], str]] = {
    "content": build_content_text,
    "structured": build_structured_text,
}


def corpus_fingerprint(
    chunks: Sequence[Mapping[str, Any]],
    *,
    text_mode: str = "structured",
) -> str:
    if text_mode not in TEXT_BUILDERS:
        raise ValueError(f"지원하지 않는 text_mode: {text_mode}")
    text_builder = TEXT_BUILDERS[text_mode]
    digest = hashlib.sha256()
    digest.update(text_mode.encode("utf-8"))
    digest.update(b"\0")
    for chunk in chunks:
        digest.update(str(chunk["chunk_id"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(text_builder(chunk).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def make_analyzer_settings(mode: str) -> dict[str, Any]:
    if mode == "standard":
        return {}
    if mode not in {"none", "discard", "mixed"}:
        raise ValueError(f"지원하지 않는 분석기 모드: {mode}")
    tokenizer_name = f"nori_{mode}_tokenizer"
    analyzer_name = f"nori_{mode}"
    return {
        "analysis": {
            "tokenizer": {
                tokenizer_name: {
                    "type": "nori_tokenizer",
                    "decompound_mode": mode,
                    "discard_punctuation": True,
                }
            },
            "analyzer": {
                analyzer_name: {
                    "type": "custom",
                    "tokenizer": tokenizer_name,
                    "filter": ["lowercase"],
                }
            },
        }
    }


def analyzer_name(mode: str) -> str:
    return "standard" if mode == "standard" else f"nori_{mode}"


def create_es_index(
    es: Any,
    *,
    index_name: str,
    analyzer_mode: str,
    chunks: Sequence[Mapping[str, Any]],
) -> None:
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
    analysis = make_analyzer_settings(analyzer_mode)
    body = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "-1",
            **analysis,
        },
        "mappings": {
            "dynamic": "strict",
            "properties": {
                "chunk_id": {"type": "keyword"},
                "business_function": {"type": "keyword"},
                "title": {"type": "text", "analyzer": analyzer_name(analyzer_mode)},
                "section_title": {"type": "text", "analyzer": analyzer_name(analyzer_mode)},
                "content": {"type": "text", "analyzer": analyzer_name(analyzer_mode)},
                "search_text": {"type": "text", "analyzer": analyzer_name(analyzer_mode)},
            },
        },
    }
    es.indices.create(index=index_name, **body)

    from elasticsearch.helpers import bulk

    actions = []
    for chunk in chunks:
        actions.append(
            {
                "_index": index_name,
                "_id": str(chunk["chunk_id"]),
                "_source": {
                    "chunk_id": str(chunk["chunk_id"]),
                    "business_function": str(chunk.get("business_function") or ""),
                    "title": str(chunk.get("title") or ""),
                    "section_title": str(chunk.get("section_title") or ""),
                    "content": str(chunk.get("content") or ""),
                    "search_text": build_search_text(chunk),
                },
            }
        )
    bulk(es, actions, chunk_size=200, request_timeout=120)
    es.indices.refresh(index=index_name)
    es.indices.put_settings(index=index_name, settings={"refresh_interval": "1s"})


def inspect_analyzer_tokens(es: Any, index_name: str, text: str) -> list[str]:
    response = es.indices.analyze(
        index=index_name,
        analyzer=es.indices.get_mapping(index=index_name)[index_name]["mappings"]["properties"]["search_text"]["analyzer"],
        text=text,
    )
    return [token["token"] for token in response["tokens"]]


def _business_filter(row: Mapping[str, Any], config: ExperimentConfig) -> str | None:
    mode = config.business_filter_mode
    if mode == "none":
        return None
    if mode == "gold":
        value = row.get("gold_business_function")
        normalized = str(value).strip() if value is not None else ""
        return BUSINESS_CODE_TO_LABEL.get(normalized, normalized) or None
    if mode == "column":
        value = row.get(config.business_filter_column)
        normalized = str(value).strip() if value is not None else ""
        return BUSINESS_CODE_TO_LABEL.get(normalized, normalized) or None
    raise ValueError(f"business_filter_mode은 none/gold/column 중 하나여야 합니다: {mode}")


class ElasticsearchRetriever:
    def __init__(
        self,
        es: Any,
        *,
        index_name: str,
        method: str,
        field_boosts: Mapping[str, float] | None = None,
    ) -> None:
        if method not in {"bm25", "bm25f"}:
            raise ValueError(method)
        self.es = es
        self.index_name = index_name
        self.method = method
        self.field_boosts = dict(field_boosts or {"title": 2.0, "section_title": 1.5, "content": 1.0})

    def search(self, question: str, *, top_k: int, business_function: str | None = None) -> list[dict[str, Any]]:
        if self.method == "bm25":
            query: dict[str, Any] = {"match": {"search_text": {"query": question, "operator": "or"}}}
        else:
            fields = [f"{field}^{boost:g}" for field, boost in self.field_boosts.items()]
            query = {
                "combined_fields": {
                    "query": question,
                    "fields": fields,
                    "operator": "or",
                }
            }
        if business_function:
            query = {
                "bool": {
                    "must": [query],
                    "filter": [{"term": {"business_function": business_function}}],
                }
            }
        response = self.es.search(
            index=self.index_name,
            query=query,
            size=top_k,
            request_cache=False,
            _source=False,
        )
        return [
            {"chunk_id": hit["_id"], "score": float(hit["_score"])}
            for hit in response["hits"]["hits"]
        ]


def _l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms != 0)


@dataclass
class BGEDocumentRepresentations:
    chunk_ids: list[str]
    business_functions: list[str]
    dense: np.ndarray
    lexical_weights: list[dict[str, float]]
    model_name: str
    max_length: int
    corpus_sha256: str
    text_mode: str


def encode_bge_documents(
    chunks: Sequence[Mapping[str, Any]],
    *,
    model: Any,
    model_name: str,
    max_length: int,
    batch_size: int,
    cache_dir: Path,
    text_mode: str,
) -> BGEDocumentRepresentations:
    if text_mode not in TEXT_BUILDERS:
        raise ValueError(f"지원하지 않는 text_mode: {text_mode}")
    cache_dir.mkdir(parents=True, exist_ok=True)
    fingerprint = corpus_fingerprint(chunks, text_mode=text_mode)
    cache_key = hashlib.sha256(
        f"{model_name}|{max_length}|{text_mode}|{fingerprint}".encode("utf-8")
    ).hexdigest()[:20]
    metadata_path = cache_dir / f"{cache_key}_metadata.json"
    dense_path = cache_dir / f"{cache_key}_dense.npy"
    sparse_path = cache_dir / f"{cache_key}_sparse.jsonl"

    if metadata_path.exists() and dense_path.exists() and sparse_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        dense = np.load(dense_path)
        lexical_weights = read_jsonl(sparse_path)
        weights = [{str(k): float(v) for k, v in row["weights"].items()} for row in lexical_weights]
        return BGEDocumentRepresentations(
            chunk_ids=list(metadata["chunk_ids"]),
            business_functions=list(metadata["business_functions"]),
            dense=_l2_normalize(np.asarray(dense, dtype=np.float32)),
            lexical_weights=weights,
            model_name=metadata["model_name"],
            max_length=int(metadata["max_length"]),
            corpus_sha256=metadata["corpus_sha256"],
            text_mode=metadata["text_mode"],
        )

    text_builder = TEXT_BUILDERS[text_mode]
    texts = [text_builder(chunk) for chunk in chunks]
    encoded = model.encode(
        texts,
        batch_size=batch_size,
        max_length=max_length,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense = _l2_normalize(np.asarray(encoded["dense_vecs"], dtype=np.float32))
    weights = [
        {str(token_id): float(weight) for token_id, weight in record.items()}
        for record in encoded["lexical_weights"]
    ]
    metadata = {
        "model_name": model_name,
        "max_length": max_length,
        "corpus_sha256": fingerprint,
        "text_mode": text_mode,
        "chunk_ids": [str(chunk["chunk_id"]) for chunk in chunks],
        "business_functions": [str(chunk.get("business_function") or "") for chunk in chunks],
    }
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    np.save(dense_path, dense)
    with sparse_path.open("w", encoding="utf-8") as handle:
        for chunk, record in zip(chunks, weights):
            handle.write(
                json.dumps({"chunk_id": str(chunk["chunk_id"]), "weights": record}, ensure_ascii=False)
                + "\n"
            )
    return BGEDocumentRepresentations(
        chunk_ids=metadata["chunk_ids"],
        business_functions=metadata["business_functions"],
        dense=dense,
        lexical_weights=weights,
        model_name=model_name,
        max_length=max_length,
        corpus_sha256=fingerprint,
        text_mode=text_mode,
    )


def build_sparse_matrix(
    lexical_weights: Sequence[Mapping[str, float]],
) -> tuple[sparse.csr_matrix, dict[str, int]]:
    token_ids = sorted({str(token) for row in lexical_weights for token in row})
    token_to_column = {token: index for index, token in enumerate(token_ids)}
    row_indices: list[int] = []
    column_indices: list[int] = []
    values: list[float] = []
    for row_index, weights in enumerate(lexical_weights):
        for token, weight in weights.items():
            row_indices.append(row_index)
            column_indices.append(token_to_column[str(token)])
            values.append(float(weight))
    matrix = sparse.csr_matrix(
        (values, (row_indices, column_indices)),
        shape=(len(lexical_weights), len(token_ids)),
        dtype=np.float32,
    )
    return matrix, token_to_column


class BGERetriever:
    def __init__(
        self,
        *,
        model: Any,
        documents: BGEDocumentRepresentations,
        mode: str,
        max_length: int,
    ) -> None:
        if mode not in {"dense", "sparse"}:
            raise ValueError(mode)
        self.model = model
        self.documents = documents
        self.mode = mode
        self.max_length = max_length
        self.sparse_matrix, self.token_to_column = build_sparse_matrix(documents.lexical_weights)
        self.business_array = np.asarray(documents.business_functions)

    def _query_dense(self, question: str) -> np.ndarray:
        output = self.model.encode(
            [question],
            batch_size=1,
            max_length=self.max_length,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        vector = np.asarray(output["dense_vecs"], dtype=np.float32)
        return _l2_normalize(vector)[0]

    def _query_sparse(self, question: str) -> sparse.csr_matrix:
        output = self.model.encode(
            [question],
            batch_size=1,
            max_length=self.max_length,
            return_dense=False,
            return_sparse=True,
            return_colbert_vecs=False,
        )
        weights = output["lexical_weights"][0]
        columns: list[int] = []
        values: list[float] = []
        for token, weight in weights.items():
            column = self.token_to_column.get(str(token))
            if column is not None:
                columns.append(column)
                values.append(float(weight))
        rows = np.zeros(len(columns), dtype=np.int32)
        return sparse.csr_matrix(
            (values, (rows, columns)),
            shape=(1, self.sparse_matrix.shape[1]),
            dtype=np.float32,
        )

    def search(self, question: str, *, top_k: int, business_function: str | None = None) -> list[dict[str, Any]]:
        if self.mode == "dense":
            query = self._query_dense(question)
            scores = self.documents.dense @ query
        else:
            query = self._query_sparse(question)
            scores = (self.sparse_matrix @ query.T).toarray().ravel()
        if business_function:
            scores = scores.copy()
            scores[self.business_array != business_function] = -np.inf
        finite = np.flatnonzero(np.isfinite(scores))
        if finite.size == 0:
            return []
        order = finite[np.argsort(scores[finite])[::-1]][:top_k]
        return [
            {"chunk_id": self.documents.chunk_ids[index], "score": float(scores[index])}
            for index in order
        ]


class CachedRetriever:
    def __init__(self, rankings: Mapping[str, Sequence[Mapping[str, Any]]]) -> None:
        self.rankings = {str(key): [dict(item) for item in value] for key, value in rankings.items()}

    def search(self, question: str, *, top_k: int, business_function: str | None = None) -> list[dict[str, Any]]:
        del business_function
        return [dict(item) for item in self.rankings[str(question)][:top_k]]


def collect_rankings(
    *,
    retriever: Any,
    eval_df: pd.DataFrame,
    config: ExperimentConfig,
    depth: int,
) -> dict[str, list[dict[str, Any]]]:
    """Hybrid 튜닝용 기본 검색기 순위를 질문별로 한 번만 계산한다."""
    rankings: dict[str, list[dict[str, Any]]] = {}
    for record in eval_df.to_dict(orient="records"):
        question = str(record["question"]).strip()
        if question in rankings:
            continue
        business_function = _business_filter(record, config)
        rankings[question] = retriever.search(
            question,
            top_k=depth,
            business_function=business_function,
        )
    return rankings


class HybridRRFRetriever:
    def __init__(
        self,
        dense_retriever: Any,
        sparse_retriever: Any,
        *,
        candidate_depth: int = 50,
        rrf_k: int = 60,
        dense_weight: float = 0.5,
        sparse_weight: float = 0.5,
    ) -> None:
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.candidate_depth = candidate_depth
        self.rrf_k = rrf_k
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight

    def search(self, question: str, *, top_k: int, business_function: str | None = None) -> list[dict[str, Any]]:
        dense = self.dense_retriever.search(
            question, top_k=self.candidate_depth, business_function=business_function
        )
        sparse_hits = self.sparse_retriever.search(
            question, top_k=self.candidate_depth, business_function=business_function
        )
        fused: dict[str, float] = defaultdict(float)
        for rank, item in enumerate(dense, start=1):
            fused[str(item["chunk_id"])] += self.dense_weight / (self.rrf_k + rank)
        for rank, item in enumerate(sparse_hits, start=1):
            fused[str(item["chunk_id"])] += self.sparse_weight / (self.rrf_k + rank)
        ranked = sorted(fused.items(), key=lambda item: (-item[1], item[0]))[:top_k]
        return [{"chunk_id": chunk_id, "score": float(score)} for chunk_id, score in ranked]


def reciprocal_rank(retrieved: Sequence[str], relevant: set[str], k: int) -> float:
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0


def average_precision_at_k(retrieved: Sequence[str], relevant: set[str], k: int) -> float:
    if not relevant:
        return float("nan")
    hits = 0
    total = 0.0
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        if chunk_id in relevant:
            hits += 1
            total += hits / rank
    return total / min(len(relevant), k)


def ndcg_at_k(
    retrieved: Sequence[str],
    primary: set[str],
    supporting: set[str],
    all_gold: set[str],
    k: int,
) -> float:
    relevance = {chunk_id: 1 for chunk_id in all_gold}
    for chunk_id in supporting & all_gold:
        relevance[chunk_id] = 1
    for chunk_id in primary & all_gold:
        relevance[chunk_id] = 2
    if not relevance:
        return float("nan")

    def dcg(values: Sequence[int]) -> float:
        return sum(
            (2**value - 1) / math.log2(rank + 1)
            for rank, value in enumerate(values, start=1)
        )

    actual = [relevance.get(chunk_id, 0) for chunk_id in retrieved[:k]]
    ideal = sorted(relevance.values(), reverse=True)[:k]
    denominator = dcg(ideal)
    return dcg(actual) / denominator if denominator else 0.0


def retrieval_metrics(
    retrieved: Sequence[str],
    *,
    primary: set[str],
    supporting: set[str],
    all_gold: set[str],
    evidence_requirement: str,
    multi_chunk_required: bool,
    complete_max_required: int,
) -> dict[str, float]:
    relevant = set(all_gold)
    top3 = list(retrieved[:3])
    top5 = list(retrieved[:5])
    top5_set = set(top5)
    hits5 = len(top5_set & relevant)
    recall5 = hits5 / len(relevant) if relevant else float("nan")
    primary_hits5 = len(top5_set & primary)
    primary_recall5 = (
        primary_hits5 / len(primary)
        if primary
        else float("nan")
    )
    precision5 = hits5 / 5
    useful_precision5 = (
        hits5 / len(top5)
        if top5
        else 0.0
    )
    f1 = (
        2 * useful_precision5 * recall5
        / (useful_precision5 + recall5)
        if useful_precision5 + recall5 > 0
        else 0.0
    )

    if evidence_requirement == "PRIMARY_ONLY":
        required = set(primary)
    elif evidence_requirement == "PRIMARY_PLUS_SUPPORT":
        required = set(primary) | set(supporting)
    elif evidence_requirement == "ALL":
        required = set(all_gold)
    else:
        required = set()

    complete_applicable = (
        bool(required)
        and (multi_chunk_required or len(required) > 1)
        and len(required) <= complete_max_required
    )
    complete = (
        float(required.issubset(top5_set))
        if complete_applicable
        else float("nan")
    )
    return {
        "hit_at_3": float(bool(set(top3) & relevant)),
        "recall_at_5": recall5,
        "primary_recall_at_5": primary_recall5,
        "mrr_at_10": reciprocal_rank(retrieved, relevant, 10),
        "ap_at_10": average_precision_at_k(retrieved, relevant, 10),
        "complete_at_5": complete,
        "ndcg_at_5": ndcg_at_k(
            retrieved,
            primary,
            supporting,
            all_gold,
            5,
        ),
        "precision_at_5": precision5,
        "useful_precision_at_5": useful_precision5,
        "f1_at_5": f1,
    }


def default_token_counter(text: str) -> int:
    try:
        import tiktoken

        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))
    except Exception:
        return max(1, math.ceil(len(text.encode("utf-8")) / 3))


def context_text(
    retrieved_ids: Sequence[str],
    chunks_by_id: Mapping[str, Mapping[str, Any]],
    *,
    top_k: int,
) -> str:
    blocks: list[str] = []
    for rank, chunk_id in enumerate(retrieved_ids[:top_k], start=1):
        chunk = chunks_by_id[chunk_id]
        blocks.append(
            "\n".join(
                [
                    f"[근거 {rank}]",
                    f"chunk_id: {chunk_id}",
                    build_search_text(chunk),
                ]
            )
        )
    return "\n\n".join(blocks)


def evaluate_retriever(
    *,
    retriever_name: str,
    retriever: Any,
    eval_df: pd.DataFrame,
    chunks: Sequence[Mapping[str, Any]],
    config: ExperimentConfig,
    token_counter: Callable[[str], int] = default_token_counter,
) -> tuple[pd.DataFrame, dict[str, list[dict[str, Any]]]]:
    chunks_by_id = {str(chunk["chunk_id"]): chunk for chunk in chunks}
    output_rows: list[dict[str, Any]] = []
    rankings: dict[str, list[dict[str, Any]]] = {}

    if config.final_top_k != 5:
        raise ValueError("이 실험의 final_top_k는 5로 고정해야 합니다.")
    if config.evaluation_depth < 10:
        raise ValueError("MRR@10/MAP@10 계산을 위해 evaluation_depth는 10 이상이어야 합니다.")

    for record in eval_df.to_dict(orient="records"):
        question = str(record["question"]).strip()
        business_function = _business_filter(record, config)
        repeat_results: list[tuple[float, list[dict[str, Any]]]] = []
        for _ in range(config.latency_repeats):
            started = time.perf_counter()
            hits = retriever.search(
                question,
                top_k=config.evaluation_depth,
                business_function=business_function,
            )
            elapsed_ms = (time.perf_counter() - started) * 1000
            repeat_results.append((elapsed_ms, hits))
        repeat_results.sort(key=lambda item: item[0])
        latency_ms, hits = repeat_results[len(repeat_results) // 2]
        hits = hits[: config.evaluation_depth]
        rankings[question] = [dict(item) for item in hits]

        evaluation_ids = [str(item["chunk_id"]) for item in hits]
        evaluation_scores = [float(item["score"]) for item in hits]
        final_hits = hits[: config.final_top_k]
        retrieved_ids = [str(item["chunk_id"]) for item in final_hits]
        retrieved_scores = [float(item["score"]) for item in final_hits]
        primary = set(record["gold_primary_chunk_ids"])
        supporting = set(record["gold_supporting_chunk_ids"])
        all_gold = set(record["gold_chunk_ids"])
        metrics = retrieval_metrics(
            evaluation_ids,
            primary=primary,
            supporting=supporting,
            all_gold=all_gold,
            evidence_requirement=str(
                record.get("gold_evidence_requirement", "")
            ).strip().upper(),
            multi_chunk_required=str(
                record.get("multi_chunk_required", "")
            ).strip().upper() == "Y",
            complete_max_required=config.complete_max_required,
        )
        assembled_context = context_text(
            retrieved_ids, chunks_by_id, top_k=config.context_top_k
        )
        context_tokens = token_counter(assembled_context)

        base = {
            key: value
            for key, value in record.items()
            if key
            not in {
                "retrieved_chunk_ids",
                "retrieved_scores",
                "evaluation_ranked_chunk_ids",
                "evaluation_ranked_scores",
                *METRIC_COLUMNS,
                "latency_ms",
                "query_embedding_cache_hit",
                "context_tokens",
                "context_limit_exceeded",
                "retriever",
            }
        }
        for column in (
            "gold_source_urls",
            "gold_primary_chunk_ids",
            "gold_supporting_chunk_ids",
            "gold_chunk_ids",
        ):
            base[column] = json.dumps(base[column], ensure_ascii=False)
        output_rows.append(
            {
                **base,
                "retriever": retriever_name,
                "retrieved_chunk_ids": json.dumps(retrieved_ids, ensure_ascii=False),
                "retrieved_scores": json.dumps(retrieved_scores, ensure_ascii=False),
                "evaluation_ranked_chunk_ids": json.dumps(evaluation_ids, ensure_ascii=False),
                "evaluation_ranked_scores": json.dumps(evaluation_scores, ensure_ascii=False),
                **metrics,
                "latency_ms": latency_ms,
                "context_tokens": context_tokens,
                "context_limit_exceeded": int(context_tokens > config.context_budget_tokens),
            }
        )
    return pd.DataFrame(output_rows), rankings


def summarize_results(
    question_results: pd.DataFrame,
    *,
    group_column: str | None = None,
) -> pd.DataFrame:
    if group_column is not None and group_column not in question_results.columns:
        return pd.DataFrame()
    groups: Iterable[tuple[Any, pd.DataFrame]]
    if group_column is None:
        groups = [("ALL", question_results)]
    else:
        groups = question_results.groupby(group_column, dropna=False, sort=True)

    rows: list[dict[str, Any]] = []
    for group_value, frame in groups:
        latency = frame["latency_ms"].astype(float)
        row = {
            "group_name": group_column or "overall",
            "group_value": group_value,
            "retriever": frame["retriever"].iloc[0],
            "question_count": len(frame),
            "complete_applicable_count": int(frame["complete_at_5"].notna().sum()),
            "hit_at_3": frame["hit_at_3"].mean(),
            "recall_at_5": frame["recall_at_5"].mean(),
            "primary_recall_at_5": frame["primary_recall_at_5"].mean(),
            "mrr_at_10": frame["mrr_at_10"].mean(),
            "map_at_10": frame["ap_at_10"].mean(),
            "complete_at_5": frame["complete_at_5"].mean(),
            "ndcg_at_5": frame["ndcg_at_5"].mean(),
            "precision_at_5": frame["precision_at_5"].mean(),
            "useful_precision_at_5": frame["useful_precision_at_5"].mean(),
            "f1_at_5": frame["f1_at_5"].mean(),
            "latency_ms_mean": latency.mean(),
            "latency_ms_p50": latency.quantile(0.50),
            "latency_ms_p95": latency.quantile(0.95),
            "context_limit_exceeded_rate": frame["context_limit_exceeded"].mean(),
            "context_tokens_mean": frame["context_tokens"].mean(),
        }
        rows.append(row)
    return pd.DataFrame(rows)


def select_rows_for_selection(
    results: pd.DataFrame,
    *,
    config: ExperimentConfig,
) -> tuple[pd.DataFrame, str]:
    if config.split_column in results.columns:
        mask = results[config.split_column].astype(str).str.lower().eq(config.selection_split.lower())
        if mask.any():
            return results.loc[mask].copy(), config.selection_split
    return results.copy(), "all(no split column or matching split)"


def choose_best(
    method_results: Mapping[str, pd.DataFrame],
    *,
    config: ExperimentConfig,
) -> tuple[str, pd.DataFrame, str]:
    rows: list[dict[str, Any]] = []
    selection_scope = ""
    for method, results in method_results.items():
        selected, selection_scope = select_rows_for_selection(results, config=config)
        summary = summarize_results(selected).iloc[0].to_dict()
        rows.append({"method": method, **summary})
    leaderboard = pd.DataFrame(rows)
    if config.selection_metric not in leaderboard.columns:
        raise ValueError(f"선정 지표가 없습니다: {config.selection_metric}")
    leaderboard = leaderboard.sort_values(
        by=[
            config.selection_metric,
            "recall_at_5",
            "mrr_at_10",
            "latency_ms_p50",
            "method",
        ],
        ascending=[False, False, False, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    return str(leaderboard.iloc[0]["method"]), leaderboard, selection_scope


def write_method_outputs(
    *,
    output_root: Path,
    stage_name: str,
    method_name: str,
    question_results: pd.DataFrame,
    domain_column: str = "gold_business_function",
) -> dict[str, Path]:
    method_dir = output_root / stage_name / method_name
    method_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "question_results": method_dir / "question_results.csv",
        "summary_overall": method_dir / "summary_overall.csv",
        "summary_by_domain": method_dir / "summary_by_domain.csv",
    }
    question_results.to_csv(paths["question_results"], index=False, encoding="utf-8-sig")
    summarize_results(question_results).to_csv(
        paths["summary_overall"], index=False, encoding="utf-8-sig"
    )
    summarize_results(question_results, group_column=domain_column).to_csv(
        paths["summary_by_domain"], index=False, encoding="utf-8-sig"
    )
    return paths


def write_selection_log(
    path: Path,
    *,
    stage: str,
    winner: str,
    selection_scope: str,
    config: ExperimentConfig,
    leaderboard: pd.DataFrame,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "stage": stage,
        "winner": winner,
        "selection_scope": selection_scope,
        "selection_metric": config.selection_metric,
        "tie_break_order": ["recall_at_5 desc", "mrr_at_10 desc", "latency_ms_p50 asc", "method asc"],
        "config": asdict(config),
        "leaderboard": leaderboard.replace({np.nan: None}).to_dict(orient="records"),
    }
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def zip_results(output_root: Path, destination: Path) -> Path:
    if destination.exists():
        destination.unlink()
    shutil.make_archive(str(destination.with_suffix("")), "zip", output_root)
    return destination


## 3. 입력 파일과 실험 설정

Colab에 다음 두 파일을 업로드합니다.

1. `KDIC_output(1).zip`
2. `Evaluation_DataSet_v3(1).xlsx`

노트북은 파일명 뒤의 `(1)`, `(2)` 등을 자동 인식합니다.
`추가 질문 시트`와 `URL 자동생성 로그`는 평가 입력에 합치지 않습니다.
`summary_by_domain.csv`나 기존 `question_results.csv`도 입력하지 않습니다.


In [ ]:
from pathlib import Path
import shutil


def resolve_uploaded_file(
    preferred: Path,
    patterns: tuple[str, ...],
) -> Path:
    if preferred.exists():
        return preferred

    matches = []
    for pattern in patterns:
        matches.extend(preferred.parent.glob(pattern))
    matches = sorted(
        {path.resolve() for path in matches if path.is_file()},
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if matches:
        selected = matches[0]
        if len(matches) > 1:
            print("입력 후보가 여러 개라 가장 최근 파일을 선택합니다.")
            for candidate in matches:
                print(" -", candidate)
        return selected
    return preferred


CONTENT_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

DATA_ZIP_PATH = resolve_uploaded_file(
    CONTENT_ROOT / "KDIC_output.zip",
    ("KDIC_output*.zip", "*KDIC_output*.zip"),
)
EVAL_XLSX_PATH = resolve_uploaded_file(
    CONTENT_ROOT / "Evaluation_DataSet_v3.xlsx",
    (
        "Evaluation_DataSet_v3*.xlsx",
        "*Evaluation*DataSet*v3*.xlsx",
    ),
)

WORK_DIR = Path("./kdic_retrieval_work")
OUTPUT_ROOT = Path("./kdic_retrieval_results_5stage_dual_hybrid")

assert DATA_ZIP_PATH.exists(), (
    "KDIC_output ZIP이 없습니다. Colab에 KDIC_output(1).zip을 업로드하세요. "
    f"현재 확인 경로: {DATA_ZIP_PATH}"
)
assert EVAL_XLSX_PATH.exists(), (
    "Evaluation DataSet v3 Excel이 없습니다. "
    "Colab에 Evaluation_DataSet_v3(1).xlsx를 업로드하세요. "
    f"현재 확인 경로: {EVAL_XLSX_PATH}"
)

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("산출물 ZIP:", DATA_ZIP_PATH)
print("평가 Excel:", EVAL_XLSX_PATH)


In [ ]:
from kdic_retrieval_eval import ExperimentConfig

CONFIG = ExperimentConfig(
    final_top_k=5,                   # 최종 반환 청크 수: 모든 방식에서 고정
    evaluation_depth=10,            # MRR@10/MAP@10 계산용 내부 평가 순위
    complete_max_required=5,
    context_budget_tokens=8_000,
    context_top_k=5,
    selection_metric="ndcg_at_5",
    selection_split="validation",
    split_column="split",
    business_filter_mode="none",
    business_filter_column="predicted_business_function",
    latency_repeats=1,
    rrf_k=60,
    allowed_review_statuses=None,    # 검수·정렬 엑셀 전체에서 Gold 무결성으로 포함 여부 결정
    evaluation_sheet_name="평가데이터셋 v3",
    gold_normalization_policy="gold_chunk_ids_authoritative",
)

BM25F_FIELD_BOOSTS = {
    "title": 2.0,
    "section_title": 1.5,
    "content": 1.0,
}

BGE_MODEL_NAME = "BAAI/bge-m3"
BGE_MAX_LENGTH = 1024
BGE_BATCH_SIZE = 4

# Dense 60~95 / Sparse 40~5, 5%p 간격
HYBRID_DENSE_WEIGHTS = [round(value / 100, 2) for value in range(60, 100, 5)]
HYBRID_WEIGHT_PAIRS = [
    (dense_weight, round(1.0 - dense_weight, 2))
    for dense_weight in HYBRID_DENSE_WEIGHTS
]

HYBRID_WEIGHT_DEPTH = 30
HYBRID_DEPTHS = [20, 30, 50]

assert HYBRID_WEIGHT_PAIRS == [
    (0.60, 0.40),
    (0.65, 0.35),
    (0.70, 0.30),
    (0.75, 0.25),
    (0.80, 0.20),
    (0.85, 0.15),
    (0.90, 0.10),
    (0.95, 0.05),
]

print("Hybrid 가중치 후보:", HYBRID_WEIGHT_PAIRS)
CONFIG


## 4. 입력 로딩, Gold 복구 및 무결성 검사

Excel의 한국어 열 이름을 평가 모듈의 표준 이름으로 매핑합니다.

- `예상질문` → `question`
- `질문ID` → `question_id`
- `도메인` → `domain`
- `질문 복잡도` → `question_complexity`
- `중요도` → `priority`

정상적인 `gold_chunk_ids`는 그대로 사용합니다. JSON 문법이 손상된 행만
Primary와 Supporting에서 복구하며, 중첩 배열은 1차원으로 평탄화합니다.
복구, Gold/등급 불일치, URL 불일치, `multi_chunk_required` 불일치는
`evaluation_dataset_validation.csv`에 남깁니다.


In [ ]:
import hashlib
import json
import warnings

import pandas as pd

from kdic_retrieval_eval import (
    corpus_fingerprint,
    load_chunks,
    load_evaluation_xlsx,
    prepare_evaluation_rows,
    validate_gold_chunk_ids,
)

chunks, chunks_path = load_chunks(DATA_ZIP_PATH, WORK_DIR)
full_eval_df = load_evaluation_xlsx(
    EVAL_XLSX_PATH,
    sheet_name=CONFIG.evaluation_sheet_name,
)
eval_df, excluded_eval_df, evaluation_validation_report = prepare_evaluation_rows(
    full_eval_df,
    chunks=chunks,
    config=CONFIG,
)
validate_gold_chunk_ids(eval_df, chunks)

if CONFIG.split_column not in eval_df.columns:
    warnings.warn(
        "평가셋에 split 열이 없습니다. 평가 포함 문항 전체로 승자를 선정하므로 "
        "결과는 예비 실험입니다. 최종 확정 실험 전 validation/test 분리를 권장합니다."
    )

def _serialize_list_columns(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    for column in (
        "gold_source_urls",
        "gold_primary_chunk_ids",
        "gold_supporting_chunk_ids",
        "gold_chunk_ids",
    ):
        if column in output.columns:
            output[column] = output[column].map(
                lambda values: json.dumps(values, ensure_ascii=False)
                if isinstance(values, list)
                else values
            )
    return output


evaluation_source_sha256 = hashlib.sha256(EVAL_XLSX_PATH.read_bytes()).hexdigest()
_serialize_list_columns(eval_df).to_csv(
    OUTPUT_ROOT / "evaluation_dataset_included.csv",
    index=False,
    encoding="utf-8-sig",
)
_serialize_list_columns(excluded_eval_df).to_csv(
    OUTPUT_ROOT / "evaluation_dataset_excluded.csv",
    index=False,
    encoding="utf-8-sig",
)
evaluation_validation_report.to_csv(
    OUTPUT_ROOT / "evaluation_dataset_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

normalization_counts = (
    evaluation_validation_report["normalization_notes"]
    .replace("", "unchanged")
    .value_counts(dropna=False)
)
exclusion_counts = (
    evaluation_validation_report.loc[
        ~evaluation_validation_report["included"], "exclusion_reason"
    ]
    .replace("", "unknown")
    .value_counts(dropna=False)
)

print("chunks.jsonl:", chunks_path)
print("청크 수:", len(chunks))
print("Excel 전체 문항:", len(full_eval_df))
print("검색 평가 포함 문항:", len(eval_df))
print("Gold 무결성으로 제외:", len(excluded_eval_df))
print("Gold 정규화 적용:", int((evaluation_validation_report["normalization_status"] == "normalized").sum()))
print("평가 Excel SHA-256:", evaluation_source_sha256)
print("Structured corpus SHA-256:", corpus_fingerprint(chunks, text_mode="structured"))
print("Content corpus SHA-256:", corpus_fingerprint(chunks, text_mode="content"))

def _required_evidence_count(row):
    requirement = str(row["gold_evidence_requirement"]).upper()
    if requirement == "PRIMARY_ONLY":
        required = row["gold_primary_chunk_ids"]
    elif requirement == "PRIMARY_PLUS_SUPPORT":
        required = list(dict.fromkeys(
            row["gold_primary_chunk_ids"]
            + row["gold_supporting_chunk_ids"]
        ))
    else:
        required = row["gold_chunk_ids"]
    return len(required)


required_counts = eval_df.apply(_required_evidence_count, axis=1)
multi_flags = (
    eval_df["multi_chunk_required"]
    .fillna("")
    .astype(str)
    .str.upper()
    .eq("Y")
)
complete_count = int(
    (
        (multi_flags | required_counts.gt(1))
        & required_counts.between(1, CONFIG.complete_max_required)
    ).sum()
)
print("Complete@5 적용 문항:", complete_count)
if complete_count < 10:
    warnings.warn(
        f"Complete@5 적용 문항이 {complete_count}개뿐이므로 참고 지표로 해석하세요."
    )

print("\n[포함 문항의 gold_review_status 분포 — 필터에는 사용하지 않음]")
display(
    eval_df["gold_review_status"].fillna("").replace("", "(blank)")
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
)
print("\n[Gold 정규화 내역]")
display(normalization_counts.rename("count").to_frame())
warning_counts = (
    evaluation_validation_report["validation_warnings"]
    .replace("", "none")
    .value_counts(dropna=False)
)
print("\n[검증 경고 — 평가는 포함되며 별도 확인 대상]")
display(warning_counts.rename("count").to_frame())
print("\n[제외 사유]")
display(exclusion_counts.rename("count").to_frame())
display(eval_df.head(3))


## 5. Elasticsearch 연결 및 Nori 실제 동작 검사


In [ ]:

from elasticsearch import Elasticsearch

auth = (ES_USERNAME, ES_PASSWORD) if ES_USERNAME and ES_PASSWORD else None
es = Elasticsearch(
    ES_URL,
    basic_auth=auth,
    request_timeout=120,
    max_retries=3,
    retry_on_timeout=True,
)

if not es.ping():
    raise RuntimeError(f"Elasticsearch 연결 실패: {ES_URL}")

plugins = es.cat.plugins(format="json")
plugin_names = {row.get("component") for row in plugins}
if "analysis-nori" not in plugin_names:
    raise RuntimeError("analysis-nori plugin이 없습니다.")

nori_probe = es.indices.analyze(
    tokenizer={"type": "nori_tokenizer", "decompound_mode": "mixed"},
    text="착오송금 반환지원",
)
nori_tokens = [item["token"] for item in nori_probe["tokens"]]
if not nori_tokens:
    raise RuntimeError("analysis-nori 토큰화 결과가 비어 있습니다.")

print("Elasticsearch:", es.info()["version"]["number"])
print("analysis-nori plugin: OK")
print("Nori 실제 토큰화: OK ->", nori_tokens)


In [ ]:

from kdic_retrieval_eval import create_es_index, inspect_analyzer_tokens

ANALYZER_MODES = {
    "BM25 + Standard": "standard",
    "BM25 + Nori-none": "none",
    "BM25 + Nori-discard": "discard",
    "BM25 + Nori-mixed": "mixed",
}
INDEX_BY_ANALYZER = {
    mode: f"kdic-retrieval-{mode.replace('_', '-')}"
    for mode in ANALYZER_MODES.values()
}

for mode, index_name in INDEX_BY_ANALYZER.items():
    print("인덱스 생성:", mode, "->", index_name)
    create_es_index(
        es,
        index_name=index_name,
        analyzer_mode=mode,
        chunks=chunks,
    )

token_rows = []
for mode, index_name in INDEX_BY_ANALYZER.items():
    for text in [
        "예금보험금과 착오송금 반환지원 신청 방법",
        "채무조정과 개인회생 신청 자격",
    ]:
        token_rows.append(
            {
                "analyzer": mode,
                "text": text,
                "tokens": inspect_analyzer_tokens(es, index_name, text),
            }
        )
display(pd.DataFrame(token_rows))


## 6. 공통 평가·저장 함수


In [ ]:

import re

import matplotlib.pyplot as plt
import seaborn as sns

from kdic_retrieval_eval import (
    choose_best,
    evaluate_retriever,
    write_method_outputs,
    write_selection_log,
)

all_method_results = {}
all_method_rankings = {}
all_retrievers = {}
stage_leaderboards = {}
selection_records = {}


def safe_name(name: str) -> str:
    return re.sub(r"[^0-9A-Za-z가-힣._+-]+", "_", name).strip("_")


def evaluate_and_save(stage: str, method: str, retriever):
    print(f"[{stage}] {method} 평가 시작")
    results, rankings = evaluate_retriever(
        retriever_name=method,
        retriever=retriever,
        eval_df=eval_df,
        chunks=chunks,
        config=CONFIG,
    )
    write_method_outputs(
        output_root=OUTPUT_ROOT,
        stage_name=stage,
        method_name=safe_name(method),
        question_results=results,
    )
    all_method_results[(stage, method)] = results
    all_method_rankings[(stage, method)] = rankings
    all_retrievers[(stage, method)] = retriever
    print(f"[{stage}] {method} 완료")
    return results


def select_stage(stage: str, results_by_method: dict):
    winner, leaderboard, scope = choose_best(results_by_method, config=CONFIG)
    stage_dir = OUTPUT_ROOT / stage
    stage_dir.mkdir(parents=True, exist_ok=True)
    leaderboard.to_csv(
        stage_dir / "leaderboard.csv",
        index=False,
        encoding="utf-8-sig",
    )
    write_selection_log(
        stage_dir / "selection_log.json",
        stage=stage,
        winner=winner,
        selection_scope=scope,
        config=CONFIG,
        leaderboard=leaderboard,
    )
    stage_leaderboards[stage] = leaderboard
    selection_records[stage] = {
        "winner": winner,
        "selection_scope": scope,
    }
    print("선정 범위:", scope)
    print("1위:", winner)
    display(
        leaderboard[
            [
                "method",
                "ndcg_at_5",
                "recall_at_5",
                "primary_recall_at_5",
                "mrr_at_10",
                "map_at_10",
                "latency_ms_p50",
                "context_limit_exceeded_rate",
            ]
        ]
    )
    return winner, leaderboard


def plot_leaderboard(stage: str):
    frame = stage_leaderboards[stage].copy()
    plot_frame = frame.melt(
        id_vars="method",
        value_vars=[
            "hit_at_3",
            "recall_at_5",
            "mrr_at_10",
            "map_at_10",
            "ndcg_at_5",
        ],
        var_name="metric",
        value_name="score",
    )
    plt.figure(figsize=(12, 5))
    sns.barplot(data=plot_frame, x="metric", y="score", hue="method")
    plt.ylim(0, 1)
    plt.title(stage)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT / stage / "metric_comparison.png",
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()


## 7. 1차 비교 — BM25 분석기 선택


In [ ]:

from kdic_retrieval_eval import ElasticsearchRetriever

stage1_name = "stage_1_analyzer"
stage1_retrievers = {
    method: ElasticsearchRetriever(
        es,
        index_name=INDEX_BY_ANALYZER[mode],
        method="bm25",
    )
    for method, mode in ANALYZER_MODES.items()
}
stage1_results = {
    method: evaluate_and_save(stage1_name, method, retriever)
    for method, retriever in stage1_retrievers.items()
}
best_analyzer_method, stage1_leaderboard = select_stage(
    stage1_name,
    stage1_results,
)
best_analyzer_mode = ANALYZER_MODES[best_analyzer_method]
best_analyzer_index = INDEX_BY_ANALYZER[best_analyzer_mode]
plot_leaderboard(stage1_name)

print("Best Analyzer:", best_analyzer_mode)


## 8. BGE-M3 문서 표현 생성

- `structured`: 제목 + 소제목 + 본문. BGE-M3 Sparse와 Dense-structured에서 사용
- `content`: 본문만 사용. Dense-content에서 사용

두 표현은 같은 BGE-M3 체크포인트와 같은 `max_length`를 사용합니다.


In [ ]:

import torch
from FlagEmbedding import BGEM3FlagModel

from kdic_retrieval_eval import BGERetriever, encode_bge_documents

use_fp16 = bool(torch.cuda.is_available())
print("CUDA:", torch.cuda.is_available(), "| use_fp16:", use_fp16)

bge_model = BGEM3FlagModel(
    BGE_MODEL_NAME,
    use_fp16=use_fp16,
)

bge_structured_documents = encode_bge_documents(
    chunks,
    model=bge_model,
    model_name=BGE_MODEL_NAME,
    max_length=BGE_MAX_LENGTH,
    batch_size=BGE_BATCH_SIZE,
    cache_dir=WORK_DIR / "bge_cache",
    text_mode="structured",
)
bge_content_documents = encode_bge_documents(
    chunks,
    model=bge_model,
    model_name=BGE_MODEL_NAME,
    max_length=BGE_MAX_LENGTH,
    batch_size=BGE_BATCH_SIZE,
    cache_dir=WORK_DIR / "bge_cache",
    text_mode="content",
)

bge_sparse_retriever = BGERetriever(
    model=bge_model,
    documents=bge_structured_documents,
    mode="sparse",
    max_length=BGE_MAX_LENGTH,
)
bge_dense_structured_retriever = BGERetriever(
    model=bge_model,
    documents=bge_structured_documents,
    mode="dense",
    max_length=BGE_MAX_LENGTH,
)
bge_dense_content_retriever = BGERetriever(
    model=bge_model,
    documents=bge_content_documents,
    mode="dense",
    max_length=BGE_MAX_LENGTH,
)

print("Structured Dense shape:", bge_structured_documents.dense.shape)
print("Content Dense shape:", bge_content_documents.dense.shape)
print("Structured Sparse docs:", len(bge_structured_documents.lexical_weights))


## 9. 2차 비교 — 최적 Sparse 구조 선택


In [ ]:

stage2_name = "stage_2_sparse"

bm25_best = ElasticsearchRetriever(
    es,
    index_name=best_analyzer_index,
    method="bm25",
)
bm25f_best = ElasticsearchRetriever(
    es,
    index_name=best_analyzer_index,
    method="bm25f",
    field_boosts=BM25F_FIELD_BOOSTS,
)

stage2_retrievers = {
    f"BM25 + {best_analyzer_mode}": bm25_best,
    f"BM25F + {best_analyzer_mode}": bm25f_best,
    "BGE-M3 Sparse-structured": bge_sparse_retriever,
}
stage2_results = {
    method: evaluate_and_save(stage2_name, method, retriever)
    for method, retriever in stage2_retrievers.items()
}
best_sparse_method, stage2_leaderboard = select_stage(
    stage2_name,
    stage2_results,
)
best_sparse_retriever = stage2_retrievers[best_sparse_method]
plot_leaderboard(stage2_name)

print("Best Sparse:", best_sparse_method)


## 10. 3차 비교 — 최적 Dense 입력 선택


In [ ]:

stage3_name = "stage_3_dense"
stage3_retrievers = {
    "BGE-M3 Dense-content": bge_dense_content_retriever,
    "BGE-M3 Dense-structured": bge_dense_structured_retriever,
}
stage3_results = {
    method: evaluate_and_save(stage3_name, method, retriever)
    for method, retriever in stage3_retrievers.items()
}
best_dense_method, stage3_leaderboard = select_stage(
    stage3_name,
    stage3_results,
)
best_dense_retriever = stage3_retrievers[best_dense_method]
plot_leaderboard(stage3_name)

print("Best Dense:", best_dense_method)


## 11. Dual Hybrid 튜닝용 기본 순위 캐시

Hybrid의 Dense 축은 `BGE-M3 Dense-structured`로 고정합니다.
Sparse 축은 다음 두 계열을 독립적으로 캐시하고 평가합니다.

1. `BGE-M3 Sparse-structured`
2. `BM25 + Nori-none/discard 중 1차 평가 우수 방식`

BM25 계열을 전체 1차 승자로 무조건 고정하지 않고, 사용자가 비교 대상으로 지정한
`Nori-none`과 `Nori-discard` 사이에서 동일 선정 기준으로 더 좋은 방식을 선택합니다.


In [ ]:
from kdic_retrieval_eval import CachedRetriever, collect_rankings

MAX_HYBRID_DEPTH = max(max(HYBRID_DEPTHS), HYBRID_WEIGHT_DEPTH)

# 사용자가 Hybrid Dense 축으로 지정한 구조화 Dense를 명시적으로 고정합니다.
HYBRID_DENSE_METHOD = "BGE-M3 Dense-structured"
hybrid_dense_retriever = bge_dense_structured_retriever

# BM25 Hybrid 축은 Nori-none과 Nori-discard 중 동일 선정 기준에서 더 높은 방식을 사용합니다.
bm25_hybrid_candidate_methods = [
    "BM25 + Nori-none",
    "BM25 + Nori-discard",
]
bm25_hybrid_board = stage1_leaderboard[
    stage1_leaderboard["method"].isin(bm25_hybrid_candidate_methods)
].copy()
if bm25_hybrid_board.empty:
    raise RuntimeError("BM25 Hybrid 후보인 Nori-none/discard 평가 결과가 없습니다.")

# stage1_leaderboard는 이미 nDCG@5 → Recall@5 → MRR@10 → 지연시간으로 정렬되어 있습니다.
BM25_HYBRID_METHOD = str(bm25_hybrid_board.iloc[0]["method"])
BM25_HYBRID_MODE = ANALYZER_MODES[BM25_HYBRID_METHOD]
bm25_hybrid_retriever = stage1_retrievers[BM25_HYBRID_METHOD]

BGE_SPARSE_FAMILY = "BGE-M3 Sparse"
BM25_SPARSE_FAMILY = f"BM25 Nori-{BM25_HYBRID_MODE}"

hybrid_sparse_families = {
    BGE_SPARSE_FAMILY: {
        "sparse_method": "BGE-M3 Sparse-structured",
        "retriever": bge_sparse_retriever,
    },
    BM25_SPARSE_FAMILY: {
        "sparse_method": BM25_HYBRID_METHOD,
        "retriever": bm25_hybrid_retriever,
    },
}

hybrid_dense_rankings = collect_rankings(
    retriever=hybrid_dense_retriever,
    eval_df=eval_df,
    config=CONFIG,
    depth=MAX_HYBRID_DEPTH,
)
cached_hybrid_dense = CachedRetriever(hybrid_dense_rankings)

hybrid_sparse_rankings_by_family = {}
cached_hybrid_sparse_by_family = {}
for family, spec in hybrid_sparse_families.items():
    rankings = collect_rankings(
        retriever=spec["retriever"],
        eval_df=eval_df,
        config=CONFIG,
        depth=MAX_HYBRID_DEPTH,
    )
    hybrid_sparse_rankings_by_family[family] = rankings
    cached_hybrid_sparse_by_family[family] = CachedRetriever(rankings)

print("Hybrid Dense:", HYBRID_DENSE_METHOD)
print("BM25 Hybrid 후보 비교:")
display(
    bm25_hybrid_board[
        ["method", "ndcg_at_5", "recall_at_5", "mrr_at_10", "latency_ms_p50"]
    ]
)
print("BM25 Hybrid 선택:", BM25_HYBRID_METHOD)
print("Dense 캐시 질문 수:", len(hybrid_dense_rankings))
for family, rankings in hybrid_sparse_rankings_by_family.items():
    print(f"{family} 캐시 질문 수:", len(rankings))


## 12. 4-1차 — Dual Hybrid 가중치 비교, depth=30 고정

두 Hybrid 계열에 동일한 8개 가중치를 적용합니다.

- `Dense-structured + BGE-M3 Sparse`: 8개
- `Dense-structured + BM25 Nori-none/discard Best`: 8개

총 16개 조합을 수치화하며, 전체 1위뿐 아니라 **계열별 1위**를 각각 선정합니다.


In [ ]:
from kdic_retrieval_eval import HybridRRFRetriever


def make_hybrid_method_name(
    family: str,
    dense_weight: float,
    sparse_weight: float,
    candidate_depth: int,
) -> str:
    return (
        f"Hybrid[{family}] "
        f"D{dense_weight:.2f} S{sparse_weight:.2f} "
        f"depth{candidate_depth}"
    )


stage41_name = "stage_4_1_dual_hybrid_weights"
stage41_retrievers = {}
stage41_configs = {}

for family, family_spec in hybrid_sparse_families.items():
    for dense_weight, sparse_weight in HYBRID_WEIGHT_PAIRS:
        method = make_hybrid_method_name(
            family,
            dense_weight,
            sparse_weight,
            HYBRID_WEIGHT_DEPTH,
        )
        stage41_configs[method] = {
            "hybrid_family": family,
            "dense_method": HYBRID_DENSE_METHOD,
            "sparse_method": family_spec["sparse_method"],
            "dense_weight": dense_weight,
            "sparse_weight": sparse_weight,
            "candidate_depth": HYBRID_WEIGHT_DEPTH,
        }
        stage41_retrievers[method] = HybridRRFRetriever(
            cached_hybrid_dense,
            cached_hybrid_sparse_by_family[family],
            candidate_depth=HYBRID_WEIGHT_DEPTH,
            rrf_k=CONFIG.rrf_k,
            dense_weight=dense_weight,
            sparse_weight=sparse_weight,
        )

expected_method_count = len(hybrid_sparse_families) * len(HYBRID_WEIGHT_PAIRS)
assert len(stage41_retrievers) == expected_method_count == 16

stage41_results = {
    method: evaluate_and_save(stage41_name, method, retriever)
    for method, retriever in stage41_retrievers.items()
}
best_weight_method_overall, stage41_leaderboard = select_stage(
    stage41_name,
    stage41_results,
)

# 설정값을 leaderboard에 붙여 대시보드와 CSV에서 바로 비교할 수 있게 합니다.
for column in [
    "hybrid_family",
    "dense_method",
    "sparse_method",
    "dense_weight",
    "sparse_weight",
    "candidate_depth",
]:
    stage41_leaderboard[column] = stage41_leaderboard["method"].map(
        lambda method: stage41_configs[method][column]
    )

stage_leaderboards[stage41_name] = stage41_leaderboard
stage41_dir = OUTPUT_ROOT / stage41_name
stage41_leaderboard.to_csv(
    stage41_dir / "leaderboard.csv",
    index=False,
    encoding="utf-8-sig",
)
stage41_leaderboard.to_csv(
    stage41_dir / "hybrid_weight_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

best_weight_methods_by_family = {}
best_weight_configs_by_family = {}
family_winner_rows = []
for family in hybrid_sparse_families:
    family_board = stage41_leaderboard[
        stage41_leaderboard["hybrid_family"] == family
    ]
    if family_board.empty:
        raise RuntimeError(f"Hybrid 계열 결과가 없습니다: {family}")
    best_method = str(family_board.iloc[0]["method"])
    best_weight_methods_by_family[family] = best_method
    best_weight_configs_by_family[family] = stage41_configs[best_method]
    family_winner_rows.append(family_board.iloc[0])

stage41_family_winners = pd.DataFrame(family_winner_rows).reset_index(drop=True)
stage41_family_winners.to_csv(
    stage41_dir / "family_winners.csv",
    index=False,
    encoding="utf-8-sig",
)

print("4-1차 전체 1위:", best_weight_method_overall)
print("4-2차로 전달할 계열별 Best 가중치:")
display(
    stage41_family_winners[
        [
            "hybrid_family",
            "sparse_method",
            "dense_weight",
            "sparse_weight",
            "ndcg_at_5",
            "recall_at_5",
            "primary_recall_at_5",
            "mrr_at_10",
            "complete_at_5",
        ]
    ]
)

# 가중치 변화가 지표에 미치는 영향을 계열별 선 그래프로 저장합니다.
weight_plot_metrics = [
    "ndcg_at_5",
    "recall_at_5",
    "mrr_at_10",
    "complete_at_5",
]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, metric in zip(axes.flat, weight_plot_metrics):
    for family in hybrid_sparse_families:
        subset = stage41_leaderboard[
            stage41_leaderboard["hybrid_family"] == family
        ].sort_values("dense_weight")
        axis.plot(
            subset["dense_weight"],
            subset[metric],
            marker="o",
            label=family,
        )
    axis.set_title(metric)
    axis.set_xlabel("Dense weight")
    axis.set_ylabel("Score")
    axis.set_xticks(HYBRID_DENSE_WEIGHTS)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle("Dual Hybrid weight comparison (depth=30)")
fig.tight_layout()
fig.savefig(
    stage41_dir / "metric_comparison.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 13. 4-2차 — 계열별 Best 가중치 × depth 20/30/50

4-1차의 전체 상위 N개를 그대로 가져오면 한 Sparse 계열이 모두 점유할 수 있습니다.
따라서 각 계열의 Best 가중치를 하나씩 유지하고, 후보 깊이만 20·30·50으로 비교합니다.

총 비교 수는 `2개 Hybrid 계열 × 3개 depth = 6개`입니다.


In [ ]:
stage42_name = "stage_4_2_dual_hybrid_depth"
stage42_retrievers = {}
stage42_configs = {}

for family, weight_config in best_weight_configs_by_family.items():
    dense_weight = weight_config["dense_weight"]
    sparse_weight = weight_config["sparse_weight"]
    for candidate_depth in HYBRID_DEPTHS:
        method = make_hybrid_method_name(
            family,
            dense_weight,
            sparse_weight,
            candidate_depth,
        )
        stage42_configs[method] = {
            **weight_config,
            "candidate_depth": candidate_depth,
        }
        stage42_retrievers[method] = HybridRRFRetriever(
            cached_hybrid_dense,
            cached_hybrid_sparse_by_family[family],
            candidate_depth=candidate_depth,
            rrf_k=CONFIG.rrf_k,
            dense_weight=dense_weight,
            sparse_weight=sparse_weight,
        )

expected_depth_method_count = len(hybrid_sparse_families) * len(HYBRID_DEPTHS)
assert len(stage42_retrievers) == expected_depth_method_count == 6

stage42_results = {
    method: evaluate_and_save(stage42_name, method, retriever)
    for method, retriever in stage42_retrievers.items()
}
best_depth_method_overall, stage42_leaderboard = select_stage(
    stage42_name,
    stage42_results,
)

for column in [
    "hybrid_family",
    "dense_method",
    "sparse_method",
    "dense_weight",
    "sparse_weight",
    "candidate_depth",
]:
    stage42_leaderboard[column] = stage42_leaderboard["method"].map(
        lambda method: stage42_configs[method][column]
    )

stage_leaderboards[stage42_name] = stage42_leaderboard
stage42_dir = OUTPUT_ROOT / stage42_name
stage42_leaderboard.to_csv(
    stage42_dir / "leaderboard.csv",
    index=False,
    encoding="utf-8-sig",
)
stage42_leaderboard.to_csv(
    stage42_dir / "hybrid_depth_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

best_hybrid_methods_by_family = {}
best_hybrid_configs_by_family = {}
family_depth_winner_rows = []
for family in hybrid_sparse_families:
    family_board = stage42_leaderboard[
        stage42_leaderboard["hybrid_family"] == family
    ]
    if family_board.empty:
        raise RuntimeError(f"Hybrid depth 결과가 없습니다: {family}")
    best_method = str(family_board.iloc[0]["method"])
    best_hybrid_methods_by_family[family] = best_method
    best_hybrid_configs_by_family[family] = stage42_configs[best_method]
    family_depth_winner_rows.append(family_board.iloc[0])

stage42_family_winners = pd.DataFrame(family_depth_winner_rows).reset_index(drop=True)
stage42_family_winners.to_csv(
    stage42_dir / "family_winners.csv",
    index=False,
    encoding="utf-8-sig",
)

print("4-2차 전체 1위:", best_depth_method_overall)
print("5차로 전달할 계열별 Best Hybrid:")
display(
    stage42_family_winners[
        [
            "hybrid_family",
            "sparse_method",
            "dense_weight",
            "sparse_weight",
            "candidate_depth",
            "ndcg_at_5",
            "recall_at_5",
            "primary_recall_at_5",
            "mrr_at_10",
            "complete_at_5",
            "latency_ms_p50",
        ]
    ]
)

# Depth별 변화도 계열별로 저장합니다.
depth_plot_metrics = ["ndcg_at_5", "recall_at_5", "mrr_at_10"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, metric in zip(axes, depth_plot_metrics):
    for family in hybrid_sparse_families:
        subset = stage42_leaderboard[
            stage42_leaderboard["hybrid_family"] == family
        ].sort_values("candidate_depth")
        axis.plot(
            subset["candidate_depth"],
            subset[metric],
            marker="o",
            label=family,
        )
    axis.set_title(metric)
    axis.set_xlabel("Candidate depth")
    axis.set_ylabel("Score")
    axis.set_xticks(HYBRID_DEPTHS)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle("Dual Hybrid candidate-depth comparison")
fig.tight_layout()
fig.savefig(
    stage42_dir / "metric_comparison.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 14. 5차 비교 — 단일 검색기와 계열별 Best Hybrid 최종 선택

최종 비교에서는 캐시가 아니라 실제 검색기를 호출해 end-to-end 질의 지연시간을 측정합니다.

비교 대상:

1. BGE-M3 Dense-structured를 포함한 3차 Best Dense
2. BGE-M3 Sparse-structured
3. BM25 Nori-none/discard 중 Hybrid용 Best BM25
4. Dense-structured + BGE-M3 Sparse의 Best Hybrid
5. Dense-structured + BM25의 Best Hybrid


In [ ]:
stage5_name = "stage_5_final_dual_hybrid"

hybrid_live_by_family = {}
for family, config in best_hybrid_configs_by_family.items():
    sparse_retriever = hybrid_sparse_families[family]["retriever"]
    hybrid_live_by_family[family] = HybridRRFRetriever(
        hybrid_dense_retriever,
        sparse_retriever,
        candidate_depth=config["candidate_depth"],
        rrf_k=CONFIG.rrf_k,
        dense_weight=config["dense_weight"],
        sparse_weight=config["sparse_weight"],
    )

stage5_retrievers = {
    f"Best Dense | {best_dense_method}": best_dense_retriever,
    "BGE Sparse | BGE-M3 Sparse-structured": bge_sparse_retriever,
    f"BM25 Lexical | {BM25_HYBRID_METHOD}": bm25_hybrid_retriever,
    (
        "Hybrid-BGE | "
        f"{best_hybrid_methods_by_family[BGE_SPARSE_FAMILY]}"
    ): hybrid_live_by_family[BGE_SPARSE_FAMILY],
    (
        "Hybrid-BM25 | "
        f"{best_hybrid_methods_by_family[BM25_SPARSE_FAMILY]}"
    ): hybrid_live_by_family[BM25_SPARSE_FAMILY],
}

stage5_results = {
    method: evaluate_and_save(stage5_name, method, retriever)
    for method, retriever in stage5_retrievers.items()
}
best_final_method, stage5_leaderboard = select_stage(
    stage5_name,
    stage5_results,
)
plot_leaderboard(stage5_name)

print("최종 검색 방식:", best_final_method)


## 15. 결과 검증 및 ZIP 패키징


In [ ]:
import importlib.metadata
from dataclasses import asdict
from datetime import datetime, timezone

from kdic_retrieval_eval import zip_results

# 모든 방식의 사용자 반환 청크가 최대 5개인지 검증
for (stage, method), frame in all_method_results.items():
    lengths = frame["retrieved_chunk_ids"].map(lambda value: len(json.loads(value)))
    if (lengths > CONFIG.final_top_k).any():
        raise AssertionError(f"Top 5 초과 결과 발견: {stage} / {method}")

all_leaderboards = []
for stage, leaderboard in stage_leaderboards.items():
    current = leaderboard.copy()
    current.insert(0, "stage", stage)
    all_leaderboards.append(current)
consolidated = pd.concat(all_leaderboards, ignore_index=True)
consolidated.to_csv(
    OUTPUT_ROOT / "all_stage_leaderboards.csv",
    index=False,
    encoding="utf-8-sig",
)

hybrid_selection_by_family = {
    family: {
        "weight_stage_method": best_weight_methods_by_family[family],
        "final_hybrid_method": best_hybrid_methods_by_family[family],
        **best_hybrid_configs_by_family[family],
        "rrf_k": CONFIG.rrf_k,
    }
    for family in hybrid_sparse_families
}

final_selection = {
    "best_analyzer": {
        "method": best_analyzer_method,
        "analyzer_mode": best_analyzer_mode,
    },
    "best_sparse_overall": best_sparse_method,
    "best_dense": best_dense_method,
    "hybrid_dense_fixed": HYBRID_DENSE_METHOD,
    "bm25_hybrid_selection": {
        "candidate_methods": bm25_hybrid_candidate_methods,
        "selected_method": BM25_HYBRID_METHOD,
        "selected_analyzer_mode": BM25_HYBRID_MODE,
    },
    "hybrid_by_family": hybrid_selection_by_family,
    "final_winner": best_final_method,
}
(OUTPUT_ROOT / "final_selection.json").write_text(
    json.dumps(final_selection, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "chunk_count": len(chunks),
    "evaluation_question_count": len(eval_df),
    "excluded_question_count": len(excluded_eval_df),
    "evaluation_source_file": EVAL_XLSX_PATH.name,
    "evaluation_source_sha256": evaluation_source_sha256,
    "evaluation_sheet_name": CONFIG.evaluation_sheet_name,
    "evaluation_normalized_question_count": int((evaluation_validation_report["normalization_status"] == "normalized").sum()),
    "evaluation_review_status_filter_applied": CONFIG.allowed_review_statuses is not None,
    "evaluation_normalization_counts": {str(key): int(value) for key, value in normalization_counts.items()},
    "evaluation_exclusion_reason_counts": {str(key): int(value) for key, value in exclusion_counts.items()},
    "structured_corpus_sha256": corpus_fingerprint(chunks, text_mode="structured"),
    "content_corpus_sha256": corpus_fingerprint(chunks, text_mode="content"),
    "config": asdict(CONFIG),
    "final_output_chunk_count": CONFIG.final_top_k,
    "metric_evaluation_depth": CONFIG.evaluation_depth,
    "hybrid_rrf_k": CONFIG.rrf_k,
    "hybrid_dense_method": HYBRID_DENSE_METHOD,
    "hybrid_sparse_families": {
        family: spec["sparse_method"]
        for family, spec in hybrid_sparse_families.items()
    },
    "hybrid_weight_pairs": HYBRID_WEIGHT_PAIRS,
    "hybrid_weight_depth": HYBRID_WEIGHT_DEPTH,
    "hybrid_depths": HYBRID_DEPTHS,
    "hybrid_weight_experiment_count": len(stage41_retrievers),
    "hybrid_depth_experiment_count": len(stage42_retrievers),
    "bm25f_field_boosts": BM25F_FIELD_BOOSTS,
    "bge_model_name": BGE_MODEL_NAME,
    "bge_max_length": BGE_MAX_LENGTH,
    "selections": selection_records,
    "final_selection": final_selection,
    "versions": {
        "elasticsearch_server": es.info()["version"]["number"],
        "elasticsearch_python": importlib.metadata.version("elasticsearch"),
        "FlagEmbedding": importlib.metadata.version("FlagEmbedding"),
        "transformers": importlib.metadata.version("transformers"),
        "numpy": importlib.metadata.version("numpy"),
        "pandas": importlib.metadata.version("pandas"),
    },
}
(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

zip_path = zip_results(
    OUTPUT_ROOT,
    Path("./KDIC_retrieval_5stage_dual_hybrid_results.zip"),
)

print("1차 Best Analyzer:", best_analyzer_mode)
print("2차 Best Sparse Overall:", best_sparse_method)
print("3차 Best Dense:", best_dense_method)
print("Hybrid Dense 고정:", HYBRID_DENSE_METHOD)
print("Hybrid BM25 선택:", BM25_HYBRID_METHOD)
for family in hybrid_sparse_families:
    print(f"4차 {family} Best:", best_hybrid_methods_by_family[family])
print("5차 최종 승자:", best_final_method)
print("최종 반환 청크 수:", CONFIG.final_top_k)
print("결과 ZIP:", zip_path.resolve())

display(
    stage5_leaderboard[
        [
            "method",
            "hit_at_3",
            "recall_at_5",
            "primary_recall_at_5",
            "mrr_at_10",
            "map_at_10",
            "complete_at_5",
            "ndcg_at_5",
            "precision_at_5",
            "useful_precision_at_5",
            "f1_at_5",
            "latency_ms_mean",
            "latency_ms_p50",
            "latency_ms_p95",
            "context_limit_exceeded_rate",
        ]
    ]
)


In [ ]:

# Colab에서 결과 ZIP 자동 다운로드
if IS_COLAB:
    from google.colab import files
    files.download(str(zip_path))


## 16. 최종 방식 실패 문항 확인


In [ ]:

final_results = stage5_results[best_final_method].copy()
failure_columns = [
    column
    for column in [
        "evaluation_id",
        "question",
        "gold_business_function",
        "question_complexity",
        "gold_chunk_ids",
        "retrieved_chunk_ids",
        "evaluation_ranked_chunk_ids",
        "hit_at_3",
        "recall_at_5",
        "primary_recall_at_5",
        "mrr_at_10",
        "ndcg_at_5",
        "context_tokens",
        "context_limit_exceeded",
    ]
    if column in final_results.columns
]
failures = final_results.sort_values(
    ["hit_at_3", "recall_at_5", "ndcg_at_5", "mrr_at_10"],
    ascending=[True, True, True, True],
)
display(failures[failure_columns].head(30))


## 결과 파일 구조

```text
kdic_retrieval_results_5stage_dual_hybrid/
├─ evaluation_dataset_included.csv
├─ evaluation_dataset_excluded.csv
├─ evaluation_dataset_validation.csv
├─ stage_1_analyzer/
├─ stage_2_sparse/
├─ stage_3_dense/
├─ stage_4_1_dual_hybrid_weights/
│  ├─ leaderboard.csv
│  ├─ hybrid_weight_comparison.csv
│  ├─ family_winners.csv
│  └─ metric_comparison.png
├─ stage_4_2_dual_hybrid_depth/
│  ├─ leaderboard.csv
│  ├─ hybrid_depth_comparison.csv
│  ├─ family_winners.csv
│  └─ metric_comparison.png
├─ stage_5_final_dual_hybrid/
├─ all_stage_leaderboards.csv
├─ final_selection.json
└─ run_manifest.json
```

- `hybrid_weight_comparison.csv`: 두 Hybrid 계열 × 8개 가중치, 총 16개 결과
- `stage_4_1.../family_winners.csv`: BGE Sparse 계열과 BM25 계열의 가중치별 승자
- `hybrid_depth_comparison.csv`: 계열별 Best 가중치에 depth 20/30/50을 적용한 6개 결과
- `stage_4_2.../family_winners.csv`: 최종 비교로 전달되는 계열별 Best Hybrid
- `question_results.csv`: 문항별 Gold, 최종 Top 5, 평가용 Top 10, 지표와 지연시간
- `summary_overall.csv`: 전체 평균
- `summary_by_domain.csv`: 업무별 평균

각 단계 루트에는 `leaderboard.csv`, `selection_log.json`, `metric_comparison.png`가 생성됩니다.
최종 ZIP 파일명은 `KDIC_retrieval_5stage_dual_hybrid_results.zip`입니다.
